# Day 03 — baseline финансовой тональности

## D3-01 — ML runtime, config и проверенный вход

**Цель.** Подготовить окружение и проверить вход будущего обучения.
Вход — локальный processed CSV Financial PhraseBank `sentences_75agree` и
`reports/day02/provenance.json`; выход этого пункта — DataFrame `(N, 5)`,
typed config и подтверждённый runtime. Модель пока не создана и не обучена.

**Краткая теория.** `minimal-v1` уже выполнил детерминированную очистку в Day02.
`text_clean` — строки, а не числовые признаки. TF-IDF позже обучит vocabulary/IDF
на **train**, затем LogisticRegression обучит параметры классификатора.
`label_id` — category ID: `0=negative`, `1=neutral`, `2=positive`, не сила тональности.

Будущий поток:
`processed (N, 5) → group-aware split → train TF-IDF (N_train, V) → LogReg →
validation predictions (N_valid,) → metrics`, где `V <= 5000`.
D3-02 создаст split, D3-03 — estimators, D3-04 — fit/predict.
Baseline оценивается на validation; held-out test остаётся для D5-04.

Источник: takala/financial_phrasebank; Malo et al. (2014); CC-BY-NC-SA-3.0.
Notebook работает на реальных локальных данных и не выполняет download.
Если Day02 artifacts отсутствуют, отдельная команда из корня:
`PYTHONPATH=src venv/bin/python -m finnews_sentiment.data.prepare_day02`.
Она требует локальный источник; при его отсутствии сначала используйте
отдельную команду `prepare_phrasebank --download` из AGENTS.md.

Окружение: существующий `venv/`, pins — `requirements-day03.txt`.
Для воспроизведения окружения: `venv/bin/python -m pip install -r requirements-day03.txt`.
В этом checkout все нужные packages уже установлены.

In [5]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "src/finnews_sentiment").is_dir():
            return candidate
    raise FileNotFoundError("Project root with SPEC.md and src/finnews_sentiment not found")


ROOT: Path = find_project_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.data.baseline_config import load_baseline_config
from finnews_sentiment.data.load_processed import load_processed_news
from finnews_sentiment.data.load_data import CLASS_ORDER
from finnews_sentiment.features.labels_duplicates import LABEL_MAPPING, analyze_duplicates

print(f"Interpreter: {sys.executable}")
print(f"Project root: {ROOT}")
print(f"Class order: {CLASS_ORDER}")

Interpreter: /home/mkkr/projects/Text-classification/venv/bin/python
Project root: /home/mkkr/projects/Text-classification
Class order: ('negative', 'neutral', 'positive')


### Runtime и config

Импорты подтверждают доступность ML stack без создания estimators или fit.
Нужные версии закреплены отдельно от будущего большого curriculum stack.
Config содержит реальные пути, `text_clean`/`label_id`, seed=42, split fractions,
TF-IDF/LogReg/Dummy settings и относительные output directories.

`validation_size=0.2` — доля **outer-train**, поэтому номинально группы разделятся
примерно на 64%/16%/20%; split пока не выполняется. По строкам размеры могут
отличаться из-за групп дубликатов. L2 для установленного scikit-learn 1.9.1
задаётся `l1_ratio=0.0`, без deprecated `penalty`.
Параметры зафиксированы до измерения validation metrics.

In [2]:
from dataclasses import asdict
from importlib.metadata import version
import json
import platform

import pandas as pd
import sklearn
import scipy
import joblib
import threadpoolctl
from IPython.display import display

runtime_versions = {"python": platform.python_version()}
for package in ("pandas", "numpy", "scikit-learn", "scipy", "joblib", "threadpoolctl"):
    runtime_versions[package] = version(package)
display(pd.Series(runtime_versions, name="version").to_frame())

baseline_config = load_baseline_config(ROOT / "configs/train_baseline.json")
print(json.dumps(asdict(baseline_config), ensure_ascii=False, indent=2, default=str))

,version
python,3.14.7
pandas,3.0.6
numpy,2.5.3
scikit-learn,1.9.1
scipy,1.18.1
joblib,1.6.0
threadpoolctl,3.7.0


{
  "policy": "baseline-v1",
  "input_path": "data/processed/financial_phrasebank_75agree.csv",
  "provenance_path": "reports/day02/provenance.json",
  "text_col": "text_clean",
  "target_col": "label_id",
  "random_state": 42,
  "test_size": 0.2,
  "validation_size": 0.2,
  "tfidf": {
    "max_features": 5000,
    "ngram_range": [
      1,
      2
    ],
    "analyzer": "word",
    "lowercase": false,
    "stop_words": null,
    "min_df": 1,
    "max_df": 1.0,
    "use_idf": true,
    "smooth_idf": true,
    "sublinear_tf": false,
    "norm": "l2",
    "token_pattern": "(?u)\\b\\w\\w+\\b",
    "strip_accents": null
  },
  "logreg": {
    "solver": "lbfgs",
    "C": 1.0,
    "l1_ratio": 0.0,
    "max_iter": 1000,
    "tol": 0.0001,
    "class_weight": null,
    "random_state": 42
  },
  "dummy": {
    "strategy": "most_frequent"
  },
  "split_output_dir": "data/processed/day03",
  "report_output_dir": "reports/day03"
}


### Вызов loader: только проверка и чтение

Loader сверяет policy, source/revision, SHA-256 config/processed CSV, схему,
порядок и типы IDs, fixed mapping, наличие трёх классов и duplicate conflicts.
Он не вызывает preprocessing повторно, не удаляет строки/дубликаты и не делает fit.

Текстовые поля читаются с `keep_default_na=False`: буквальные `NA`/`nan` остаются
текстами. Integer поля сначала проверяются как CSV literals и только затем
преобразуются в `int64`, чтобы `1.5`, boolean и пропуски не маскировались.
Все необходимые ошибки останавливают подготовку; missing artifacts не скачиваются.

Проверяем повторную загрузку и неизменность bytes доступных входных файлов.
Checksums raw/interim здесь служат проверкой неизменности, а loader связывает
processed CSV с сохранённым provenance Day02, без повторной конвертации архива.

In [3]:
import hashlib

processed_path = ROOT / baseline_config.input_path
provenance_path = ROOT / baseline_config.provenance_path
dataset_config_path = ROOT / "configs/dataset.json"
input_candidates = (
    processed_path, provenance_path, dataset_config_path,
    ROOT / "configs/train_baseline.json",
    ROOT / "data/raw/FinancialPhraseBank-v1.0.zip",
    ROOT / "data/interim/financial_phrasebank_75agree.csv",
)
input_checksums_before = {
    path: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in input_candidates if path.is_file()
}
news = load_processed_news(processed_path, provenance_path, dataset_config_path)
news_snapshot = news.copy(deep=True)
print(f"Processed DataFrame shape: {news.shape}")
print(f"Processed CSV SHA-256: {hashlib.sha256(processed_path.read_bytes()).hexdigest()}")
display(news.head())
display(news.dtypes.rename("dtype").to_frame())
class_counts = news["sentiment"].value_counts().reindex(CLASS_ORDER, fill_value=0)
display(pd.DataFrame({"count": class_counts, "share": class_counts / len(news)}))
print(f"Label mapping: {LABEL_MAPPING}")
duplicates = analyze_duplicates(news)
print(f"Raw repeats beyond first: {duplicates.raw_duplicate_rows}")
print(f"Clean repeats beyond first: {duplicates.clean_duplicate_rows}")
print(f"Conflicting groups: {duplicates.has_conflicts}")

Processed DataFrame shape: (3453, 5)
Processed CSV SHA-256: 39dda898a281dc3294be081917b0681524e73edb60f00913183c67714da0c448


,source_row_id,text,sentiment,text_clean,label_id
0,0,"According to Gran , the company has no plans t...",neutral,"according to gran , the company has no plans t...",1
1,1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2
2,2,"For the last quarter of 2010 , Componenta 's n...",positive,"for the last quarter of 2010 , componenta 's n...",2
3,3,"In the third quarter of 2010 , net sales incre...",positive,"in the third quarter of 2010 , net sales incre...",2
4,4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2


,dtype
source_row_id,int64
text,str
sentiment,str
text_clean,str
label_id,int64


,count,share
sentiment,,
negative,420,0.121633
neutral,2146,0.621489
positive,887,0.256878


Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Raw repeats beyond first: 5
Clean repeats beyond first: 5
Conflicting groups: False


### Проверка результата и интерпретация

Counts описывают входную таблицу, **не качество классификации**.
Большая доля `neutral` объясняет, зачем позже сравнивать macro F1 с accuracy
и с DummyClassifier. Повторяющиеся тексты сохранены и требуют group-aware split:
одна raw/clean группа целиком попадёт в один split в D3-02.

Повторная загрузка ниже должна сохранить значения/порядок DataFrame и bytes
доступных файлов. На D3-01 ещё нет train/validation/test, vocabulary, predictions
и измеренных metrics; готовность окружения не доказывает качество модели.

In [4]:
reloaded_news = load_processed_news(processed_path, provenance_path, dataset_config_path)
pd.testing.assert_frame_equal(news, reloaded_news)
pd.testing.assert_frame_equal(news, news_snapshot)
input_checksums_after = {
    path: hashlib.sha256(path.read_bytes()).hexdigest() for path in input_checksums_before
}
assert input_checksums_after == input_checksums_before
print("Repeated load: identical values, dtypes and row order")
print(f"Input checksums unchanged: {len(input_checksums_before)} files")
print("D3-01: verified input/runtime; split, estimators and fit are pending")

Repeated load: identical values, dtypes and row order
Input checksums unchanged: 6 files
D3-01: verified input/runtime; split, estimators and fit are pending


### Самопроверка

1. Почему `positive=2` не означает «вдвое больше neutral=1»?
2. Почему `text_clean` готов к чтению, но TF-IDF ещё нуждается в `fit`?
3. Какие несоответствия обнаруживает provenance, а какие требуют проверки таблицы?
4. Почему сохранение всех repeats требует группового split?
5. Почему validation_size=0.2 не даёт 20% всей таблицы?
6. Почему counts классов и успешный импорт sklearn ничего не говорят о macro F1?

Следующий пункт — D3-02; его реализация на этом checkpoint не начата.

## D3-02 — воспроизводимый stratified group-aware split

**Цель.** Получить manifest `(N, 3)` с `source_row_id, group_id, split`,
три таблицы `(N_split, 5)` и summary в памяти. Вход — проверенный `news` из D3-01.

**Краткая теория.** Train нужен для будущего fit TF-IDF/LogReg, validation —
для сравнения вариантов, held-out test — для единственной итоговой оценки
после заморозки решения в D5-04. Здесь test проверяем только структурно.
Обычный stratified row split может положить повторяющееся предложение в
train и validation: модель встретит тот же текст, и оценка станет слишком лёгкой.
Поэтому единица разделения — группа точного совпадения `text_clean`.
Same-label duplicates сохраняем; fuzzy matching не выполняем.

`group_id` — минимальный `source_row_id`; pandas index не является ID.
Алгоритм `group-stratified-v1`: группы в порядке ID → stratified test=20%,
seed=42 → снова упорядочить outer-train → stratified validation=20% от остатка,
seed=42 → назначить split всем строкам группы, сохраняя исходный порядок.
Это примерно 64/16/20% **групп**; rounding и размеры repeats меняют доли строк.
Нехватка групп/классов и conflicts останавливают split без fallback.

**Вызов кода.** Используем config, зафиксированный до получения ML metrics.


In [5]:
from finnews_sentiment.data.split_data import SPLIT_ORDER, split_processed_news

news_splits = split_processed_news(
    news, test_size=baseline_config.test_size,
    validation_size=baseline_config.validation_size,
    random_state=baseline_config.random_state,
)
split_manifest = news_splits.manifest
train_news = news_splits.splits["train"]
validation_news = news_splits.splits["validation"]
test_news = news_splits.splits["test"]
print(f"Algorithm: {news_splits.summary.algorithm}; seed: {news_splits.summary.random_state}")
print(f"Manifest: {split_manifest.shape}; source rows: {news_splits.summary.rows}; groups: {news_splits.summary.groups}")
split_sizes = pd.DataFrame([
    {"split": name, "rows": stats.rows, "row_share": stats.row_share,
     "groups": stats.groups, "group_share": stats.group_share}
    for name, stats in news_splits.summary.splits.items()
]).set_index("split")
display(split_sizes)
display(split_manifest.head())


Algorithm: group-stratified-v1; seed: 42
Manifest: (3453, 3); source rows: 3453; groups: 3448


,rows,row_share,groups,group_share
split,,,,
train,2210,0.640023,2206,0.639791
validation,553,0.160151,552,0.160093
test,690,0.199826,690,0.200116


,source_row_id,group_id,split
0,0,0,test
1,1,1,train
2,2,2,validation
3,3,3,validation
4,4,4,train


### Counts и доли классов

Для каждой строки таблицы ниже `row_class_share = row_count / N_split`,
а `group_class_share = group_count / G_split`. Stratification распределяет
группы приблизительно по классам; сохранённые repeats меняют веса строк.
Counts test — структурная проверка, не метрики модели и не анализ его ошибок.


In [6]:
split_balance = pd.DataFrame([
    {"split": name, "sentiment": label,
     "row_count": stats.row_class_counts[label],
     "row_class_share": stats.row_class_shares[label],
     "group_count": stats.group_class_counts[label],
     "group_class_share": stats.group_class_shares[label]}
    for name, stats in news_splits.summary.splits.items() for label in CLASS_ORDER
])
display(split_balance.set_index(["split", "sentiment"]))


row_count  row_class_share  group_count  \
split      sentiment                                            
train      negative         269         0.121719          269   
           neutral         1374         0.621719         1370   
           positive         567         0.256561          567   
validation negative          67         0.121157           67   
           neutral          344         0.622061          343   
           positive         142         0.256781          142   
test       negative          84         0.121739           84   
           neutral          428         0.620290          428   
           positive         178         0.257971          178   

                      group_class_share  
split      sentiment                     
train      negative            0.121940  
           neutral             0.621034  
           positive            0.257026  
validation negative            0.121377  
           neutral             0.621377  
           positive            0.257246  
test       negative            0.121739  
           neutral             0.620290  
           positive            0.257971

### Проверка результата

Проверим полное покрытие IDs, отсутствие пересечений по IDs/raw/clean texts,
целостность групп, минимальные group IDs, сохранение порядка и всех строк.
Повторный вызов должен дать тот же manifest и summary, не изменяя inputs.


In [7]:
from itertools import combinations

assert split_manifest["source_row_id"].tolist() == news["source_row_id"].tolist()
assert split_manifest["source_row_id"].is_unique
assert sum(len(frame) for frame in news_splits.splits.values()) == len(news)
assert split_manifest.groupby("group_id")["split"].nunique().eq(1).all()
minimum_ids = news.groupby("text_clean")["source_row_id"].min()
assert split_manifest["group_id"].tolist() == news["text_clean"].map(minimum_ids).tolist()
for left, right in combinations(SPLIT_ORDER, 2):
    for column in ("source_row_id", "text", "text_clean"):
        assert set(news_splits.splits[left][column]).isdisjoint(news_splits.splits[right][column])
for name, frame in news_splits.splits.items():
    assert set(frame["sentiment"]) == set(CLASS_ORDER)
    ids = split_manifest.loc[split_manifest["split"] == name, "source_row_id"]
    pd.testing.assert_frame_equal(frame, news.loc[news["source_row_id"].isin(ids)])
repeated_splits = split_processed_news(
    news, test_size=baseline_config.test_size,
    validation_size=baseline_config.validation_size,
    random_state=baseline_config.random_state,
)
pd.testing.assert_frame_equal(split_manifest, repeated_splits.manifest)
assert news_splits.summary == repeated_splits.summary
pd.testing.assert_frame_equal(news, news_snapshot)
assert input_checksums_before == {
    path: hashlib.sha256(path.read_bytes()).hexdigest() for path in input_checksums_before
}
print("Verified: complete coverage, intact groups, disjoint IDs/raw/clean texts, original row order")
print("Repeated split identical; source DataFrame and input bytes unchanged")
print(f"Disjointness: {news_splits.summary.disjoint}")


Verified: complete coverage, intact groups, disjoint IDs/raw/clean texts, original row order
Repeated split identical; source DataFrame and input bytes unchanged
Disjointness: {'source_row_id': True, 'text': True, 'text_clean': True}


### Интерпретация и ограничения

Фактические размеры train/validation/test: **2210/553/690 строк**,
**2206/552/690 групп**. Всего 3453 строки и 3448 групп: все 5 repeats сохранены.
Четыре repeat rows оказались в train, одна — в validation. Разницы между долями
строк и групп небольшие, но это результат данных, а не гарантия алгоритма.
Во всех splits три класса, neutral остаётся большинством; точных пересечений нет.

Группы защищают от leakage через точные совпадения, но не через парафразы.
Одинаковые dataset/config/runtime воспроизводят manifest; другой dataset или
версия sklearn могут изменить назначение даже при том же seed.
Manifest и summary пока в памяти; сохранение reference artifacts — D3-06.

### Самопроверка

1. Почему stratify по строкам недостаточно при same-label duplicates?
2. Чем число групп отличается от числа строк, и к чему относятся 64/16/20%?
3. Почему outer-train нужно снова сортировать перед вторым split?
4. Что означает `group_id`, и почему это не pandas index?
5. Почему class counts test сейчас разрешены, а test metrics отложены?
6. Почему split не должен молча менять seed при недостатке групп?

Следующий пункт — D3-03: создание необученных estimators.
